# 24. Contextual Retrieval RAG (Anthropic Pattern)

## Overview
Contextual Retrieval prepends chunk-specific explanatory context before embedding, reducing retrieval failures by 49-67%.

**Pattern Source**: Anthropic (Sept 2024)  
**Verified Results**: 49% failure reduction (Contextual Embeddings + BM25), 67% with reranking

## Architecture
```
Document → Chunk → Generate Context → Prepend → Embed → Store
                                                         ↓
Query → Contextual Embedding + BM25 → Rerank (top 150 → 20) → LLM
```

## Key Innovation
- Generate 50-100 token context explaining what each chunk is about
- Prepend context before embedding (improves semantic matching)
- Hybrid retrieval: Contextual embeddings + Contextual BM25
- Two-stage reranking for precision

## When to Use
- High-precision requirements (legal, medical, compliance)
- Technical documentation with specific identifiers
- Retrieval failure rate > 5%
- Knowledge bases with complex domain concepts

In [ ]:
import boto3
import json
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import time
from typing import List, Dict
from rank_bm25 import BM25Okapi
import numpy as np

## Configuration

In [ ]:
# AWS Configuration
region = 'us-east-1'
bedrock_runtime = boto3.client('bedrock-runtime', region_name=region)
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=region)

# OpenSearch Configuration
host = 'YOUR_OPENSEARCH_ENDPOINT'
index_name = 'contextual-retrieval-rag'

# Model IDs
CLAUDE_MODEL_ID = 'anthropic.claude-sonnet-4-20250514-v1:0'
EMBED_MODEL_ID = 'amazon.titan-embed-text-v2:0'

# OpenSearch client setup
service = 'aoss'
credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    service,
    session_token=credentials.token
)

os_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

## Step 1: Context Generation
Generate explanatory context for each chunk using Claude

In [ ]:
def generate_chunk_context(document_title: str, document_content: str, chunk_content: str) -> str:
    """
    Generate contextual explanation for a chunk using Claude.
    This is the core innovation of Contextual Retrieval.
    """
    prompt = f"""Here is a document:

<document>
Title: {document_title}
{document_content}
</document>

Here is a chunk extracted from the document:
<chunk>
{chunk_content}
</chunk>

Please provide a short, succinct context (50-100 tokens) to situate this chunk within the overall document. The context should help someone understand what this chunk is about without seeing the rest of the document. Focus on:
- What topic or section this belongs to
- Key entities or concepts mentioned
- Purpose of this information

Context:"""
    
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 150,
        "temperature": 0.3,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    })
    
    try:
        response = bedrock_runtime.invoke_model(
            modelId=CLAUDE_MODEL_ID,
            body=body
        )
        
        response_body = json.loads(response['body'].read())
        context = response_body['content'][0]['text'].strip()
        return context
    except Exception as e:
        print(f"Error generating context: {e}")
        return ""

# Test context generation
sample_doc_title = "AWS RAG Best Practices"
sample_doc = """RAG systems combine retrieval with generation. 
Key considerations include chunk size, embedding models, and reranking.
AWS Bedrock provides Claude models for generation and Titan for embeddings."""

sample_chunk = "AWS Bedrock provides Claude models for generation and Titan for embeddings."

context = generate_chunk_context(sample_doc_title, sample_doc, sample_chunk)
print(f"Generated Context:\n{context}\n")
print(f"Contextualized Chunk:\n{context} {sample_chunk}")

## Step 2: Document Processing with Contextual Chunking

In [ ]:
def chunk_document(text: str, chunk_size: int = 512, overlap: int = 50) -> List[str]:
    """Split document into overlapping chunks."""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.split()) > 50:  # Minimum chunk size
            chunks.append(chunk)
    
    return chunks

def process_document_with_context(doc_title: str, doc_content: str) -> List[Dict]:
    """
    Process document: chunk → generate context → create contextualized chunks.
    """
    # Step 1: Chunk the document
    chunks = chunk_document(doc_content, chunk_size=512, overlap=50)
    print(f"Created {len(chunks)} chunks from document: {doc_title}")
    
    # Step 2: Generate context for each chunk
    contextualized_chunks = []
    
    for idx, chunk in enumerate(chunks):
        # Generate context (cached to avoid redundant API calls in production)
        context = generate_chunk_context(doc_title, doc_content, chunk)
        
        # Create contextualized chunk
        contextualized_text = f"{context} {chunk}"
        
        contextualized_chunks.append({
            'chunk_id': f"{doc_title}_{idx}",
            'document_title': doc_title,
            'original_chunk': chunk,
            'context': context,
            'contextualized_chunk': contextualized_text,
            'chunk_index': idx
        })
        
        print(f"  Chunk {idx + 1}/{len(chunks)}: Generated context ({len(context.split())} tokens)")
        
        # Rate limiting to avoid throttling
        time.sleep(0.5)
    
    return contextualized_chunks

# Sample document
sample_document = {
    'title': 'RAG Production Best Practices',
    'content': """Retrieval Augmented Generation (RAG) combines retrieval with generation.
The optimal chunk size for Q&A documentation is 256-512 tokens. For legal and compliance
documents, use 512-1024 tokens. Hybrid retrieval combining embeddings with BM25 provides
better coverage than pure vector search. BM25 excels at exact keyword matching for
product codes, policy numbers, and technical identifiers. Reranking with cross-encoders
improves precision by selecting the top 20 chunks from an initial 150. The most common
production failure is over-fetching - using k=10 instead of k=3-5 dilutes relevance.
Contextual retrieval reduces failures by 49% through chunk-specific context generation."""
}

contextualized_chunks = process_document_with_context(
    sample_document['title'],
    sample_document['content']
)

print(f"\nContextualized {len(contextualized_chunks)} chunks")
print(f"\nExample contextualized chunk:\n{contextualized_chunks[0]['contextualized_chunk'][:300]}...")

## Step 3: Embedding with Context

In [ ]:
def get_embedding(text: str) -> List[float]:
    """Get embedding for contextualized chunk."""
    body = json.dumps({
        "inputText": text[:8000]  # Titan limit
    })
    
    response = bedrock_runtime.invoke_model(
        modelId=EMBED_MODEL_ID,
        body=body
    )
    
    response_body = json.loads(response['body'].read())
    return response_body['embedding']

def index_contextualized_chunks(chunks: List[Dict]):
    """Index contextualized chunks with embeddings and BM25 preparation."""
    # Create index if not exists
    if not os_client.indices.exists(index=index_name):
        index_body = {
            'settings': {
                'index': {
                    'knn': True,
                    'number_of_shards': 1,
                    'number_of_replicas': 0
                }
            },
            'mappings': {
                'properties': {
                    'chunk_id': {'type': 'keyword'},
                    'document_title': {'type': 'text'},
                    'original_chunk': {'type': 'text'},
                    'context': {'type': 'text'},
                    'contextualized_chunk': {'type': 'text'},  # For BM25
                    'embedding': {
                        'type': 'knn_vector',
                        'dimension': 1024,
                        'method': {
                            'name': 'hnsw',
                            'space_type': 'cosinesimil',
                            'engine': 'faiss'
                        }
                    },
                    'chunk_index': {'type': 'integer'}
                }
            }
        }
        os_client.indices.create(index=index_name, body=index_body)
        print(f"Created index: {index_name}")
    
    # Index each chunk
    for chunk_data in chunks:
        # Embed the contextualized chunk (this is the key innovation)
        embedding = get_embedding(chunk_data['contextualized_chunk'])
        
        document = {
            'chunk_id': chunk_data['chunk_id'],
            'document_title': chunk_data['document_title'],
            'original_chunk': chunk_data['original_chunk'],
            'context': chunk_data['context'],
            'contextualized_chunk': chunk_data['contextualized_chunk'],
            'embedding': embedding,
            'chunk_index': chunk_data['chunk_index']
        }
        
        os_client.index(
            index=index_name,
            body=document,
            id=chunk_data['chunk_id'],
            refresh=True
        )
        
        print(f"Indexed: {chunk_data['chunk_id']}")
        time.sleep(0.3)

# Index the contextualized chunks
index_contextualized_chunks(contextualized_chunks)
print(f"\nIndexed {len(contextualized_chunks)} contextualized chunks")

## Step 4: Hybrid Retrieval (Contextual Embeddings + BM25)

In [ ]:
def hybrid_contextual_retrieval(query: str, k: int = 150) -> List[Dict]:
    """
    Hybrid retrieval: Contextual embeddings (semantic) + BM25 (keyword).
    Retrieve top 150 for reranking stage.
    """
    # Step 1: Vector search with contextualized embeddings
    query_embedding = get_embedding(query)
    
    vector_query = {
        'size': k,
        'query': {
            'knn': {
                'embedding': {
                    'vector': query_embedding,
                    'k': k
                }
            }
        },
        '_source': ['chunk_id', 'document_title', 'original_chunk', 'context', 'contextualized_chunk']
    }
    
    vector_results = os_client.search(index=index_name, body=vector_query)
    vector_hits = vector_results['hits']['hits']
    
    # Step 2: BM25 search on contextualized chunks
    bm25_query = {
        'size': k,
        'query': {
            'match': {
                'contextualized_chunk': {
                    'query': query,
                    'operator': 'or'
                }
            }
        },
        '_source': ['chunk_id', 'document_title', 'original_chunk', 'context', 'contextualized_chunk']
    }
    
    bm25_results = os_client.search(index=index_name, body=bm25_query)
    bm25_hits = bm25_results['hits']['hits']
    
    # Step 3: Combine and deduplicate results
    combined_results = {}
    
    # Add vector results with scores
    for hit in vector_hits:
        chunk_id = hit['_source']['chunk_id']
        combined_results[chunk_id] = {
            'chunk_id': chunk_id,
            'document_title': hit['_source']['document_title'],
            'original_chunk': hit['_source']['original_chunk'],
            'context': hit['_source']['context'],
            'vector_score': hit['_score'],
            'bm25_score': 0
        }
    
    # Add BM25 scores
    for hit in bm25_hits:
        chunk_id = hit['_source']['chunk_id']
        if chunk_id in combined_results:
            combined_results[chunk_id]['bm25_score'] = hit['_score']
        else:
            combined_results[chunk_id] = {
                'chunk_id': chunk_id,
                'document_title': hit['_source']['document_title'],
                'original_chunk': hit['_source']['original_chunk'],
                'context': hit['_source']['context'],
                'vector_score': 0,
                'bm25_score': hit['_score']
            }
    
    # Normalize and combine scores (RRF - Reciprocal Rank Fusion)
    results_list = list(combined_results.values())
    
    # Sort by vector score to get ranks
    vector_sorted = sorted(results_list, key=lambda x: x['vector_score'], reverse=True)
    vector_ranks = {item['chunk_id']: idx + 1 for idx, item in enumerate(vector_sorted)}
    
    # Sort by BM25 score to get ranks
    bm25_sorted = sorted(results_list, key=lambda x: x['bm25_score'], reverse=True)
    bm25_ranks = {item['chunk_id']: idx + 1 for idx, item in enumerate(bm25_sorted)}
    
    # Calculate RRF scores
    k_rrf = 60  # Standard RRF constant
    for result in results_list:
        chunk_id = result['chunk_id']
        vector_rank = vector_ranks.get(chunk_id, len(results_list) + 1)
        bm25_rank = bm25_ranks.get(chunk_id, len(results_list) + 1)
        
        rrf_score = (1 / (k_rrf + vector_rank)) + (1 / (k_rrf + bm25_rank))
        result['hybrid_score'] = rrf_score
    
    # Sort by hybrid score
    results_list.sort(key=lambda x: x['hybrid_score'], reverse=True)
    
    return results_list[:k]

# Test hybrid retrieval
test_query = "What is the optimal chunk size for legal documents?"
results = hybrid_contextual_retrieval(test_query, k=10)

print(f"Query: {test_query}\n")
print(f"Retrieved {len(results)} results\n")

for idx, result in enumerate(results[:3]):
    print(f"Result {idx + 1}:")
    print(f"  Document: {result['document_title']}")
    print(f"  Context: {result['context'][:100]}...")
    print(f"  Chunk: {result['original_chunk'][:150]}...")
    print(f"  Scores - Vector: {result['vector_score']:.4f}, BM25: {result['bm25_score']:.4f}, Hybrid: {result['hybrid_score']:.6f}")
    print()

## Step 5: Reranking (Top 150 → Top 20)
Two-stage reranking for maximum precision

In [ ]:
def simple_rerank(query: str, results: List[Dict], top_k: int = 20) -> List[Dict]:
    """
    Simple reranking based on query-chunk relevance.
    In production, use Cohere reranker or cross-encoder models.
    """
    query_words = set(query.lower().split())
    
    for result in results:
        # Calculate relevance: context + original chunk
        text = f"{result['context']} {result['original_chunk']}".lower()
        text_words = set(text.split())
        
        # Word overlap score
        overlap = len(query_words.intersection(text_words))
        overlap_ratio = overlap / len(query_words) if query_words else 0
        
        # Combine with existing hybrid score
        result['rerank_score'] = result['hybrid_score'] * 0.7 + overlap_ratio * 0.3
    
    # Sort by rerank score
    results.sort(key=lambda x: x['rerank_score'], reverse=True)
    
    return results[:top_k]

# Retrieve 150 candidates, then rerank to top 20
candidates = hybrid_contextual_retrieval(test_query, k=150)
reranked_results = simple_rerank(test_query, candidates, top_k=20)

print(f"Reranked from {len(candidates)} to {len(reranked_results)} results\n")
print("Top 3 after reranking:\n")

for idx, result in enumerate(reranked_results[:3]):
    print(f"Result {idx + 1}:")
    print(f"  Document: {result['document_title']}")
    print(f"  Final Score: {result['rerank_score']:.6f}")
    print(f"  Chunk: {result['original_chunk'][:200]}...")
    print()

## Step 6: Generation with Contextualized Retrieval

In [ ]:
def generate_contextual_rag_response(query: str, top_k_final: int = 5) -> Dict:
    """
    Full contextual retrieval pipeline: Retrieve 150 → Rerank to 20 → Use top 5 for generation.
    """
    # Stage 1: Hybrid retrieval (150 candidates)
    candidates = hybrid_contextual_retrieval(query, k=150)
    print(f"Stage 1: Retrieved {len(candidates)} candidates via hybrid search")
    
    # Stage 2: Reranking (top 20)
    reranked = simple_rerank(query, candidates, top_k=20)
    print(f"Stage 2: Reranked to top {len(reranked)} results")
    
    # Stage 3: Select top k for generation
    top_results = reranked[:top_k_final]
    print(f"Stage 3: Using top {len(top_results)} results for generation\n")
    
    # Build context from original chunks (not contextualized versions)
    context_parts = []
    for idx, result in enumerate(top_results):
        context_parts.append(f"[Document {idx + 1}: {result['document_title']}]\n{result['original_chunk']}")
    
    context = "\n\n".join(context_parts)
    
    # Generate response
    prompt = f"""You are a helpful assistant. Answer the question based ONLY on the provided context.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {query}

Answer:"""
    
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "temperature": 0.7,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ]
    })
    
    response = bedrock_runtime.invoke_model(
        modelId=CLAUDE_MODEL_ID,
        body=body
    )
    
    response_body = json.loads(response['body'].read())
    answer = response_body['content'][0]['text']
    
    return {
        'query': query,
        'answer': answer,
        'sources': top_results,
        'num_candidates': len(candidates),
        'num_reranked': len(reranked),
        'num_final': len(top_results)
    }

# Test the full pipeline
response = generate_contextual_rag_response(
    "What is the optimal chunk size for legal documents?",
    top_k_final=5
)

print("=" * 80)
print("CONTEXTUAL RETRIEVAL RAG RESPONSE")
print("=" * 80)
print(f"\nQuery: {response['query']}")
print(f"\nAnswer:\n{response['answer']}")
print(f"\nPipeline Stats:")
print(f"  - Candidates retrieved: {response['num_candidates']}")
print(f"  - After reranking: {response['num_reranked']}")
print(f"  - Used for generation: {response['num_final']}")
print(f"\nTop Sources:")
for idx, source in enumerate(response['sources'][:3]):
    print(f"  {idx + 1}. {source['document_title']} (score: {source['rerank_score']:.4f})")

## Performance Comparison: Standard vs Contextual Retrieval

In [ ]:
def compare_retrieval_methods(queries: List[str]):
    """
    Compare standard RAG vs Contextual Retrieval.
    """
    results = []
    
    for query in queries:
        # Contextual retrieval (what we implemented)
        contextual_response = generate_contextual_rag_response(query, top_k_final=5)
        
        results.append({
            'query': query,
            'contextual_answer': contextual_response['answer'],
            'contextual_sources': len(contextual_response['sources'])
        })
    
    return results

# Test queries
test_queries = [
    "What is the optimal chunk size for legal documents?",
    "How does BM25 help with RAG?",
    "What causes over-fetching in production RAG systems?"
]

comparison = compare_retrieval_methods(test_queries)

print("\n" + "=" * 80)
print("COMPARISON RESULTS")
print("=" * 80)

for idx, result in enumerate(comparison):
    print(f"\nQuery {idx + 1}: {result['query']}")
    print(f"Answer: {result['contextual_answer'][:200]}...")
    print(f"Sources: {result['contextual_sources']}")
    print("-" * 80)

## Key Metrics & Benefits

### Verified Performance (Anthropic 2024)
- **Contextual Embeddings + BM25**: 49% failure reduction (5.7% → 2.9%)
- **With Reranking**: 67% failure reduction (5.7% → 1.9%)

### Cost Analysis
- **Context Generation**: $1-2 per million document tokens (one-time preprocessing)
- **Query Cost**: Similar to standard RAG (no per-query overhead)
- **ROI**: Significant for high-precision use cases (legal, medical, compliance)

### Latency Impact
- **Preprocessing**: +0.5-1s per chunk (one-time)
- **Query Latency**: +100-300ms (hybrid + reranking)
- **Two-stage reranking**: +50-150ms

### When to Use
✅ High-precision requirements  
✅ Technical documentation  
✅ Failure rate > 5%  
✅ Domain-specific jargon  

### When NOT to Use
❌ Simple Q&A with low precision needs  
❌ Frequently changing documents (context regeneration cost)  
❌ Latency-critical applications (< 500ms requirement)  
❌ Cost-constrained environments with acceptable failure rates

## Cleanup

In [ ]:
# Optional: Delete index
# os_client.indices.delete(index=index_name)
# print(f"Deleted index: {index_name}")